# 01 — Exploratory Data Analysis & Baseline Model

This notebook performs the first pass of analysis for the Credit Risk Intelligence System.

Goals:

1. Load the Home Credit training dataset
2. Understand target imbalance
3. Inspect missing values and feature types
4. Build a baseline Logistic Regression model
5. Evaluate the model using credit-risk appropriate metrics

In [ ]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

In [ ]:
df = pd.read_csv("../data/raw/application_train.csv")

df.shape

In [ ]:
df.head()

In [ ]:
df["TARGET"].value_counts()

In [ ]:
df["TARGET"].value_counts(normalize=True)

In [ ]:
df.dtypes.value_counts()

In [ ]:
target = "TARGET"

X = df.drop(columns=[target])
y = df[target]

numeric_features = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_features = X.select_dtypes(include=["object"]).columns.tolist()

len(numeric_features), len(categorical_features)

In [ ]:
missing_summary = (
    df.isnull()
    .mean()
    .sort_values(ascending=False)
    .reset_index()
)

missing_summary.columns = ["column", "missing_rate"]

missing_summary.head(25)

In [ ]:
high_missing_cols = missing_summary[missing_summary["missing_rate"] > 0.40]["column"].tolist()

len(high_missing_cols), high_missing_cols[:10]

In [ ]:
target = "TARGET"
id_column = "SK_ID_CURR"

columns_to_drop = [id_column] + high_missing_cols

X = df.drop(columns=[target] + columns_to_drop)
y = df[target]

X.shape

In [ ]:
numeric_features = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_features = X.select_dtypes(include=["object"]).columns.tolist()

print("Numeric features:", len(numeric_features))
print("Categorical features:", len(categorical_features))
print("Total features before encoding:", X.shape[1])

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training rows:", X_train.shape[0])
print("Testing rows:", X_test.shape[0])
print("Training target rate:")
print(y_train.value_counts(normalize=True))
print("Testing target rate:")
print(y_test.value_counts(normalize=True))

In [ ]:
numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

In [ ]:
logistic_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", LogisticRegression(
            max_iter=1000,
            class_weight="balanced",
            random_state=42
        ))
    ]
)

logistic_model.fit(X_train, y_train)

In [ ]:
y_pred = logistic_model.predict(X_test)
y_pred_proba = logistic_model.predict_proba(X_test)[:, 1]

In [ ]:
roc_auc = roc_auc_score(y_test, y_pred_proba)

print("ROC-AUC:", round(roc_auc, 4))
print()
print("Classification Report:")
print(classification_report(y_test, y_pred))

In [ ]:
confusion_matrix(y_test, y_pred)

In [ ]:
cm = confusion_matrix(y_test, y_pred)

cm_df = pd.DataFrame(
    cm,
    index=["Actual Non-Default", "Actual Default"],
    columns=["Predicted Non-Default", "Predicted Default"]
)

cm_df

## Baseline Model Interpretation

The Logistic Regression model is used as the first baseline because it is simple, interpretable, and useful for comparing more advanced models later.

Because the dataset is highly imbalanced, accuracy is not the main success metric. A model could achieve high accuracy by predicting most applicants as non-default.

For this project, ROC-AUC, recall for the default class, precision, and the confusion matrix are more important.